## L1-Penalized GP with Bilby

*Notes*:

- In BL, using 10,000 iterations with 1000 iterations of burn-in. 

- Gamma prior on lambda squared with r = 1, delta = 1.78


*To Do*:
- scale data, run sampler, then scale back so that priors can be similar
- run sanity checks
    - take posterior likelihood from BL, check values with likelihood compared to random values
- test different samplers
    - 0.2 < acceptance fraction < 0.5
    - look at trace plots and how many samples are needed for convergence
    - try: pyemcee

- two pieces of this analysis
    - can this method reproduce some of the main results?
    - also some of a way of getting some uncertainty analysis


- he is sending guide to running jupyter notebook on a quest server
    - quest server: something he can request

*Done*
- reduce dimensions as a sanity check





Posterior:
$$p(\beta, \tau^2, \sigma^2_N, \sigma^2_{GP}, \ell, \lambda \mid y) \propto p(y | \beta, \sigma^2_N, \sigma^2_{GP}, \ell) \times p(\beta | \sigma^2_N, \tau^2) \times \prod_{j=1}^p p(\tau_j^2 | \lambda) \times p(\sigma^2_N) \times p(\sigma^2_{GP}) \times p(\ell) \times p(\lambda)$$

\begin{align*}
& p(y, \beta, \tau^2, \sigma^2_N, \sigma^2_{GP}, \ell, \lambda) \\
&= \frac{\exp\!\left(
 -\frac12 (y - X\beta)^\top (K + \sigma^2_N I_n)^{-1} (y - X\beta)
\right)}
{\sqrt{(2\pi)^n |K + \sigma^2_N I_n|}} \\[6pt]
&\quad \times \frac{\exp\!\left(-\frac12 \beta^\top (\sigma^2_N D_\tau)^{-1} \beta \right)}
{(2\pi)^{p/2} |\sigma^2_N D_\tau|^{1/2}} \\[6pt]
&\quad \times \prod_{j=1}^p \frac{\lambda^2}{2} \exp\!\left( -\frac{\lambda^2}{2} \tau_j^2 \right) \\[6pt]
&\quad \times \frac{\beta_{\text{noise}}^{\alpha_{\text{noise}}}}{\Gamma(\alpha_{\text{noise}})}
(\sigma^2_N)^{-\alpha_{\text{noise}}-1} \exp\!\left( -\frac{\beta_{\text{noise}}}{\sigma^2_N} \right) \\[6pt]
&\quad \times \frac{\beta_{\text{GP}}^{\alpha_{\text{GP}}}}{\Gamma(\alpha_{\text{GP}})}
(\sigma^2_{GP})^{-\alpha_{\text{GP}}-1} \exp\!\left( -\frac{\beta_{\text{GP}}}{\sigma^2_{GP}} \right) \\[6pt]
&\quad \times \frac{1}{\ell \, \sigma_\ell \sqrt{2\pi}}
\exp\!\left( -\frac{(\log \ell - \mu_\ell)^2}{2\sigma_\ell^2} \right) \\[6pt]
&\quad \times \frac{b_\lambda^{a_\lambda}}{\Gamma(a_\lambda)} \lambda^{a_\lambda - 1} e^{-b_\lambda \lambda}
\end{align*}

Log Likelihoods: 

*check over with change to lambda squared not lambda*

1. GP Likelihood

$$\log p(y \mid \beta, \sigma^2_N, \sigma^2_{GP}, \ell)
= -\tfrac{n}{2}\log(2\pi)
-\tfrac{1}{2}\log\!\big| K(\ell,\sigma^2_{GP}) + \sigma^2_N I_n \big|
-\tfrac{1}{2}(y - X\beta)^\top \big( K + \sigma^2_N I_n \big)^{-1}(y - X\beta)$$


2. $\beta \mid \tau^2, \sigma^2_N$: MVN

$$\log p(\beta \mid \tau^2, \sigma^2_N) 
= -\tfrac{p}{2}\log(2\pi)
-\tfrac{1}{2}\log\!\big|\sigma^2_N D_\tau\big|
-\tfrac{1}{2}\beta^\top (\sigma^2_N D_\tau)^{-1}\beta$$

3. $\tau_j^2 \mid \lambda$: Each exponential with rate $\frac{\lambda^2}{2}$

$$\log p(\tau^2 \mid \lambda)
= \sum_{j=1}^p \left[ \log\!\Big(\tfrac{\lambda^2}{2}\Big) - \tfrac{\lambda^2}{2}\tau_j^2 \right]
= p \log\!\Big(\tfrac{\lambda^2}{2}\Big) - \tfrac{\lambda^2}{2}\sum_{j=1}^p \tau_j^2$$

4. $\sigma^2_N$: Inverse Gamma

$$\log p(\sigma^2_N)
= \alpha_{\text{noise}} \log \beta_{\text{noise}}
- \log \Gamma(\alpha_{\text{noise}})
- (\alpha_{\text{noise}} + 1)\log \sigma^2_N
- \tfrac{\beta_{\text{noise}}}{\sigma^2_N}$$

5. $\sigma^2_{GP}$: Inverse Gamma

$$\log p(\sigma^2_{GP})
= \alpha_{\text{GP}} \log \beta_{\text{GP}}
- \log \Gamma(\alpha_{\text{GP}})
- (\alpha_{\text{GP}} + 1)\log \sigma^2_{GP}
- \tfrac{\beta_{\text{GP}}}{\sigma^2_{GP}}$$

6. $\ell$: Log Normal

$$\log p(\ell)
= -\log\!\big(\ell \, \sigma_\ell \sqrt{2\pi}\big)
- \frac{(\log \ell - \mu_\ell)^2}{2\sigma_\ell^2}$$

7. $\lambda$: Gamma

$$\log p(\lambda^2)
= r \log \delta
- \log \Gamma(r)
+ (\delta - 1)\log \lambda^2
- \delta \lambda^2$$



### Unused Code

In [110]:
### --- to load simulated data ---
# sample_dat_path = "../synthetic_data/N11000_AP10_noise0.1_seed0/Size50/Rep1.csv"
# sample_dat = pd.read_csv(sample_dat_path)

# val_dat_path = "../synthetic_data/N11000_AP10_noise0.1_seed0/N11000_AP10_noise0.1_seed0_meta.json"
# with open(val_dat_path, 'r') as f:
#     val_dat = json.load(f)

# true_betas = np.array(val_dat['beta'])
# true_ells = np.array(val_dat['lengthscales'])
# active_dims = np.array(val_dat['active_indices'])
# true_sigma_noise = val_dat['noise_constant']

# X = sample_dat.iloc[:, :-1].values  #all columns except the last
# y = sample_dat.iloc[:, -1].values   #last column

# Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=22)

In [111]:
## to save time plotting


# fig = plt.Figure() # notice the capital F
# sns.pairplot(.....)
# plt.savefig(.....)
# plt.close()


# pairplot(corner = True) # only plots half of symmetrical plot


### Data

In [112]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import bilby
from bilby.core.utils import random
import json
import scipy.special
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# set up
random.seed(123)

# label = "penalizedGP"
# outdir = "outdir2"

label = "small"
outdir = "testing"

bilby.utils.check_directory_exists_and_if_not_mkdir(outdir)

In [113]:
### --- to load diabetes data ---
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
X = diabetes.data

selected_features = ['bmi', 'bp', 's1']
X = X[selected_features]

label_names = diabetes.feature_names
y = diabetes.target

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=22)

scaler = StandardScaler()
Xtrain = scaler.fit_transform(Xtrain) # fit and scale training data
Xtest = scaler.transform(Xtest) # scale test data

### MCMC

In [114]:
# helper function to compute the RBF kernel
def rbf_kernel(X, ell, sigma_gp):
    N = X.shape[0]
    K = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            diff = (X[i] - X[j]) / ell
            K[i, j] = sigma_gp**2 * np.exp(-0.5 * np.dot(diff, diff))
    return K

In [115]:
# custom inverse gamma prior for bilby
class InverseGammaPrior(bilby.core.prior.Prior):
    def __init__(self, alpha, beta, name="inverse_gamma", latex_label=None, unit=None):
        super().__init__(name=name, latex_label=latex_label, unit=unit)
        self.alpha = alpha
        self.beta = beta

    def prob(self, x):
        return stats.invgamma.pdf(x, a=self.alpha, scale=self.beta) # use built in inverse gamma pdf

    def ln_prob(self, x):
        return stats.invgamma.logpdf(x, a=self.alpha, scale=self.beta) 

In [116]:
# custom likelihood for penalized GP regression

class PenalizedGPLikelihood(bilby.Likelihood):
    def __init__(self, X, y):
        # store data
        self.X = np.asarray(X)
        self.y = np.asarray(y)

        # define parameters
        parameters = {}

        # linear coefficients (betas)
        for i in range(self.X.shape[1]):
            parameters[f"beta{i}"] = None

        # variance parameters (tau^2)
        for i in range(self.X.shape[1]):
            parameters[f"tau_sq_{i}"] = None

        # noise parameters
        parameters["sigma_noise"] = None
        parameters["sigma_gp"] = None
        
        # lengthscales (ells)
        for i in range(self.X.shape[1]):
            parameters[f"ell{i}"] = None

        # lambda (for L1 regularization)
        parameters["lambda2"] = None
        
        super().__init__(parameters=parameters)


    def log_likelihood(self):

        # extract parameters
        betas = np.array([self.parameters[f"beta{i}"] for i in range(self.X.shape[1])])
        tau_sqs = np.array([self.parameters[f"tau_sq_{i}"] for i in range(self.X.shape[1])])
        sigma_noise = self.parameters["sigma_noise"]
        sigma_gp = self.parameters["sigma_gp"]
        ells = np.array([self.parameters[f"ell{i}"] for i in range(self.X.shape[1])])
        lbda2 = self.parameters["lambda2"] # sample lambda squared
        lbda = np.sqrt(lbda2) # take square root to get lambda
        
        # calculate n and p
        n = self.X.shape[0]
        p = self.X.shape[1]

        # calculate covariance matrix C
        C = rbf_kernel(self.X, ells, sigma_gp)
        
        # calculate residuals
        residuals = self.y - self.X @ betas

        # calculate diagonal matrix D
        D = np.diag(tau_sqs)

        # --- log likelihood components ---

        # GP likelihood
        log_lik_gp = (
            -0.5 * n * np.log(2 * np.pi) 
            -0.5 * np.linalg.slogdet(C + sigma_noise**2 * np.eye(n))[1]
            -0.5 *(residuals).T @ np.linalg.solve(C + sigma_noise**2 * np.eye(n), residuals))

        # beta (vector)
        log_lik_beta = (
            -0.5 * p * np.log(2 * np.pi)
            -0.5 * np.linalg.slogdet(sigma_noise**2 * D)[1]
            -0.5 * betas.T @ np.linalg.solve(sigma_noise**2 * D, betas)
        )
        
        # ## check over this part
        # # tau_sqs (vector)
        # log_lik_tau = (
        #     p * np.log(lbda**2 / 2.0)
        #     - 0.5 * lbda**2 * np.sum(tau_sqs)
        # )

        # # sigma_noise (scalar): coming from inverse-gamma distribution with alpha=?, beta=?
        # self.alpha_noise = 1.0
        # self.beta_noise = 1.0
        # log_lik_sigma_noise = (
        #     self.alpha_noise * np.log(self.beta_noise)
        #     - scipy.special.gammaln(self.alpha_noise)
        #     - (self.alpha_noise + 1) * np.log(sigma_noise**2)
        #     - self.beta_noise / (sigma_noise**2)
        # )

        # ### switch to using custom inverse gamma prior?
        # # sigma_gp (scalar): coming from inverse-gamma distribution with alpha=?, beta=?
        # self.alpha_gp = 1.0
        # self.beta_gp = 1.0
        # log_lik_sigma_gp = (
        #     self.alpha_gp * np.log(self.beta_gp)
        #     - scipy.special.gammaln(self.alpha_gp)
        #     - (self.alpha_gp + 1) * np.log(sigma_gp)
        #     - self.beta_gp / sigma_gp
        # )


        # # ells (vector): coming from log-normal distribution with mu=0, sigma^2=1
        # self.mu_ell = 0.0
        # self.sigma_ell = 1.0
        # log_lik_ells = np.sum(
        #     -np.log(ells * self.sigma_ell * np.sqrt(2 * np.pi))
        #     - (np.log(ells) - self.mu_ell) ** 2 / (2 * self.sigma_ell**2)
        # )

        # # lbda (scalar): coming from gamma distribution
        # self.r_lambda = 1.0 # coming from BL paper
        # self.d_lambda = 1.78 # coming from BL paper
        
        # # check over with lambda squared
        # log_lik_lbda2 = (
        #     self.r_lambda * np.log(self.d_lambda)
        #     - scipy.special.gammaln(self.r_lambda)
        #     + (self.r_lambda - 1) * np.log(lbda**2)
        #     - self.d_lambda * (lbda**2)
        #     + np.log(2 * lbda)
        # )
        
        # sum up the log likelihood components
        log_likelihood = (log_lik_gp + log_lik_beta)

        return log_likelihood


In [117]:
# make priors
priors = dict()

# --- option 1: define very wide domains for all priors ---

# very wide domains for all priors because we include in the log likelihood
for i in range(Xtrain.shape[1]):
    priors[f"beta{i}"] = bilby.core.prior.Uniform(-1e3, 1e3, name=f"beta{i}") # wide
    priors[f"tau_sq_{i}"] = bilby.core.prior.LogUniform(1e-4, 1e2, name=f"tau_sq_{i}") # positive
    priors[f"ell{i}"] = bilby.core.prior.LogUniform(1e-3, 1e2, name=f"ell{i}") # positive

priors["sigma_noise"] = bilby.core.prior.LogUniform(1e-4, 1e2, name="sigma_noise")  # positive
priors["sigma_gp"] = bilby.core.prior.LogUniform(1e-4, 1e2, name="sigma_gp")  # positive
priors["lambda2"] = bilby.core.prior.Gamma(1.0, 1.78, name="lambda2") # positive


# --- option 2: define specic domains for all priors ---

# for i in range(Xtrain.shape[1]):
#     priors[f"beta{i}"] = bilby.core.prior.Normal(0, 250, f"beta{i}") # define normal priors for each beta coefficient
#     priors[f"tau_sq_{i}"] = bilby.core.prior.Exponential(0.5, f"tau{i}_sq")
#     priors[f"ell{i}"] = bilby.core.prior.LogNormal(0, 1, f"ell{i}") # define log-normal priors for each lengthscale
# # priors["sigma_noise"] = InverseGammaPrior(1, 1, "sigma_noise") 
# priors["sigma_noise"] = bilby.core.prior.LogUniform(1e-6, 1e2, name="sigma_noise")
# # priors["sigma_gp"] = InverseGammaPrior(1, 1, "sigma_gp") 
# priors["sigma_gp"] = bilby.core.prior.LogUniform(1e-6, 1e2, name="sigma_gp")  # positive
# priors["lambda"] = bilby.core.prior.Gamma(1, 1, "lambda") 


# define the likelihood function that we defined earlier
likelihood = PenalizedGPLikelihood(
    X = Xtrain,
    y = ytrain)

In [118]:
# run MCMC sampler
result = bilby.run_sampler(
    likelihood=likelihood, # likelihood function
    priors=priors, # prior distributions
    sampler="emcee", 
    nwalkers = 200,
    nsteps = 50,
    nburn = 20,
    outdir=outdir,
    label=label)

09:49 bilby INFO    : Running for label 'small', output will be saved to 'testing'
09:49 bilby INFO    : Analysis priors:
09:49 bilby INFO    : beta0=Uniform(minimum=-1000.0, maximum=1000.0, name='beta0', latex_label='beta0', unit=None, boundary=None)
09:49 bilby INFO    : tau_sq_0=LogUniform(minimum=0.0001, maximum=100.0, name='tau_sq_0', latex_label='tau_sq_0', unit=None, boundary=None)
09:49 bilby INFO    : ell0=LogUniform(minimum=0.001, maximum=100.0, name='ell0', latex_label='ell0', unit=None, boundary=None)
09:49 bilby INFO    : beta1=Uniform(minimum=-1000.0, maximum=1000.0, name='beta1', latex_label='beta1', unit=None, boundary=None)
09:49 bilby INFO    : tau_sq_1=LogUniform(minimum=0.0001, maximum=100.0, name='tau_sq_1', latex_label='tau_sq_1', unit=None, boundary=None)
09:49 bilby INFO    : ell1=LogUniform(minimum=0.001, maximum=100.0, name='ell1', latex_label='ell1', unit=None, boundary=None)
09:49 bilby INFO    : beta2=Uniform(minimum=-1000.0, maximum=1000.0, name='beta2', l

emcee: Exception while calling your likelihood function:
  params: [-323.96292617    2.78602374    1.1052459  -109.28866313    3.12575527
    6.37620421   99.65903544    1.80652866    1.99070549   43.18203349
    8.22706516    1.81510658]
  args: []
  kwargs: {}
  exception:


SystemExit: 130

/Users/liviafingerson/Library/Python/3.10/lib/python/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Analysis of Results

In [ ]:
# scale data back
Xtrain_orig = scaler.inverse_transform(Xtrain)
Xtest_orig = scaler.inverse_transform(Xtest)

In [58]:
result.plot_walkers()
# see outdir/GP_walkers.png

In [63]:
result.posterior

,beta0,tau_sq_0,ell0,beta1,tau_sq_1,ell1,beta2,tau_sq_2,ell2,beta3,...,tau_sq_28,ell28,beta29,tau_sq_29,ell29,sigma_noise,sigma_gp,lambda2,log_likelihood,log_prior
0,129.770707,2.711987,8.307222,-52.052980,6.833878,5.947182,37.528628,7.881067,7.076371,-72.644381,...,4.452063,9.503231,-172.656379,5.029684,8.092300,19.587571,6.851697,1.497823,-5087.164494,-509.580641
1,129.770707,2.711987,8.307222,-52.052980,6.833878,5.947182,37.528628,7.881067,7.076371,-72.644381,...,4.452063,9.503231,-172.656379,5.029684,8.092300,19.587571,6.851697,1.497823,-5193.493883,-505.951248
2,129.770707,2.711987,8.307222,-52.052980,6.833878,5.947182,37.528628,7.881067,7.076371,-72.644381,...,4.452063,9.503231,-172.656379,5.029684,8.092300,19.587571,6.851697,1.497823,-32433.122981,-505.103034
3,129.770707,2.711987,8.307222,-52.052980,6.833878,5.947182,37.528628,7.881067,7.076371,-72.644381,...,4.452063,9.503231,-172.656379,5.029684,8.092300,19.587571,6.851697,1.497823,-12032.395197,-505.031115
4,129.770707,2.711987,8.307222,-52.052980,6.833878,5.947182,37.528628,7.881067,7.076371,-72.644381,...,4.452063,9.503231,-172.656379,5.029684,8.092300,19.587571,6.851697,1.497823,-9240.481838,-504.993980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10995,38.424166,3.881430,8.485301,-140.813411,4.286304,14.423061,85.529046,8.216345,10.371700,-93.754547,...,0.545427,6.157842,-161.689451,6.901109,3.682341,24.457431,15.333299,1.290622,-2914.949632,-515.815835
10996,38.424166,3.881430,8.485301,-140.813411,4.286304,14.423061,85.529046,8.216345,10.371700,-93.754547,...,0.545427,6.157842,-161.689451,6.901109,3.682341,24.457431,15.333299,1.290622,-2643.504215,-495.204528
10997,38.424166,3.881430,8.485301,-140.813411,4.286304,14.423061,85.529046,8.216345,10.371700,-93.754547,...,0.545427,6.157842,-161.689451,6.901109,3.682341,24.457431,15.333299,1.290622,-2490.856739,-442.043889
10998,-16.356185,5.038102,13.878671,-8.047638,2.857461,10.992579,19.961099,5.735286,8.850259,-25.306347,...,7.467067,5.541045,-116.295342,11.287440,5.649337,28.905576,11.515009,1.376169,-2727.299102,-506.115818
